# 04 · Entrenamiento GAN: StyleGAN2-ADA condicional 128×128 (reanudable)

**Fase CRISP-DM:** 4. Modelado.

**Qué hace:** entrena StyleGAN2-ADA condicional sobre **train** (3060 imágenes, 17 clases), guardando *snapshots* en Drive cada `SNAP_KIMG` kimg.
Si la sesión se cae, **vuelve a ejecutar todo el notebook**: detecta el último snapshot y continúa desde ahí.

**Qué NO hace:** no elige el mejor checkpoint ni evalúa. Eso se hace después con validación (DINOv2), en otro notebook. Test no se usa.

**Configuración (fija para la tesis):** `--cond=1 --mirror=0 --metrics=none --cfg=auto` (batch 32), semilla 42, mismos 4 parches del notebook 03b.

**Reanudación correcta (parche 5, propio):** al continuar, el repo original reinicia el contador de kimg y la probabilidad de aumento de ADA, y acelera ADA (`ada_kimg=100`, pensado para *transfer learning*).
El parche 5 restaura el contador y la probabilidad `p` del snapshot y mantiene `ada_kimg=500`. El estado del optimizador Adam no se guarda en los snapshots del repo: se reinicia en cada reanudación (efecto transitorio; se declara como limitación).

**Duración:** 2000 kimg ≈ 14.5 h en L4 (26 s/kimg medidos en el piloto) ≈ 26 unidades.
**Antes de empezar:** GPU **L4**, versión **2026.07**; laptop sin suspensión; pestaña abierta.

## Paso 1 · Entorno

In [1]:
import sys, os, subprocess, json, shutil, hashlib, re, time, glob
from pathlib import Path
from datetime import datetime, timezone
import torch
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ SIN GPU")

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | CUDA: 12.8
GPU: NVIDIA L4


## Paso 2 · Parámetros

In [2]:
SEED = 42
TOTAL_KIMG = int(os.environ.get("TOTAL_KIMG", 2000))   # tope de entrenamiento (puede ampliarse luego reanudando)
SNAP_KIMG = int(os.environ.get("SNAP_KIMG", 100))      # cada cuánto se guarda un snapshot (≈ 43 min)
KIMG_PER_TICK = 4                                      # valor por defecto del repo
SNAP_TICKS = SNAP_KIMG // KIMG_PER_TICK
RUN_NAME = "stylegan2ada_cond128_seed42"

## Paso 3 · Drive, dataset y carpeta del entrenamiento

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()
LOGS = PROJECT_DIR / "logs"
RUN_DIR = PROJECT_DIR / "checkpoints" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG03 = json.loads((LOGS / "03_piloto.json").read_text())
SG_ZIP = LOCAL_DIR / LOG03["datasets"]["train_zip"]
if not SG_ZIP.exists():
    shutil.copy2(PROJECT_DIR / "datasets" / SG_ZIP.name, SG_ZIP)
assert sha256_file(SG_ZIP) == LOG03["datasets"]["train_sha256"], "⚠️ El dataset no coincide con el del notebook 03"
print("Dataset verificado ✔ |", "carpeta del entrenamiento:", RUN_DIR)

Mounted at /content/drive
Dataset verificado ✔ | carpeta del entrenamiento: /content/drive/MyDrive/tesis_retail/checkpoints/stylegan2ada_cond128_seed42


## Paso 4 · Repo oficial + 5 parches

In [4]:
%pip install -q click requests tqdm ninja psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 20.4 MB/s eta 0:00:00


In [5]:
SG_COMMIT = "d72cc7d041b42ec8e806021a205ed9349f87c6a4"
SG3_COMMIT = "c233a919a6faee6e36a316ddd4eddababad1adf9"
SG_DIR = LOCAL_DIR / "stylegan2-ada-pytorch"
SG3_DIR = LOCAL_DIR / "stylegan3"

def clonar(url, destino, commit):
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", url, str(destino)], check=True)
    subprocess.run(["git", "-C", str(destino), "checkout", "-q", commit], check=True)
    subprocess.run(["git", "-C", str(destino), "checkout", "-q", "--", "."], check=True)   # archivos limpios
    head = subprocess.run(["git", "-C", str(destino), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head == commit, head

clonar("https://github.com/NVlabs/stylegan2-ada-pytorch.git", SG_DIR, SG_COMMIT)
clonar("https://github.com/NVlabs/stylegan3.git", SG3_DIR, SG3_COMMIT)

def parchar(ruta, viejo, nuevo, veces=1):
    s = ruta.read_text()
    assert s.count(viejo) == veces, f"Parche no aplicable en {ruta.name}: {viejo[:60]}"
    ruta.write_text(s.replace(viejo, nuevo))

# 1) distutils (Python >= 3.12)
parchar(SG_DIR / "dnnlib" / "util.py", "from distutils.util import strtobool",
        "try:\n    from distutils.util import strtobool\nexcept ImportError:  # Python >= 3.12\n"
        "    def strtobool(v):\n        return v.lower() in ('y', 'yes', 't', 'true', 'on', '1')")
# 2) Sampler.__init__ sin argumentos
parchar(SG_DIR / "torch_utils" / "misc.py", "        super().__init__(dataset)\n", "        super().__init__()\n")
# 3) grid_sample_gradfix de StyleGAN3 sin pkg_resources
s = (SG3_DIR / "torch_utils" / "ops" / "grid_sample_gradfix.py").read_text()
s = s.replace("from pkg_resources import parse_version\n", "")
s = s.replace("_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
              "_TV = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])\n_use_pytorch_1_11_api = _TV >= (1, 11)")
s = s.replace("_use_pytorch_1_12_api = parse_version(torch.__version__) >= parse_version('1.12.0a') # Allow prerelease builds of 1.12",
              "_use_pytorch_1_12_api = _TV >= (1, 12)")
assert "parse_version" not in s
(SG_DIR / "torch_utils" / "ops" / "grid_sample_gradfix.py").write_text(s)
# 4) custom_ops: usar el módulo que devuelve cpp_extension.load
co = SG_DIR / "torch_utils" / "custom_ops.py"
parchar(co, "            torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,",
            "            module = torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,")
parchar(co, "            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)",
            "            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)")
parchar(co, "        module = importlib.import_module(module_name)\n", "")
# 5) reanudación: restaurar contador de kimg y p de ADA; mantener ada_kimg=500
tl = SG_DIR / "training" / "training_loop.py"
parchar(tl, "        augment_pipe.p.copy_(torch.as_tensor(augment_p))\n",
            "        augment_pipe.p.copy_(torch.as_tensor(augment_p))\n"
            "        if (resume_pkl is not None) and (resume_data.get('augment_pipe') is not None):  # parche 5\n"
            "            augment_pipe.p.copy_(resume_data['augment_pipe'].p)\n"
            "            print(f'ADA p restaurado: {float(augment_pipe.p):.4f}')\n")
parchar(tl, "    cur_nimg = 0\n", "    cur_nimg = int(os.environ.get('SG_RESUME_KIMG', '0')) * 1000  # parche 5\n")
parchar(SG_DIR / "train.py", "        args.ada_kimg = 100 # make ADA react faster at the beginning",
        "        if not os.environ.get('SG_RESUME_KIMG'): args.ada_kimg = 100 # parche 5: solo en transfer learning")
print("5 parches aplicados ✔")

os.environ["TORCH_EXTENSIONS_DIR"] = str(LOCAL_DIR / "torch_extensions")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"

5 parches aplicados ✔


## Paso 5 · ¿Empezar o reanudar?

In [6]:
def snapshots():
    out = []
    for p in RUN_DIR.glob("*/network-snapshot-*.pkl"):
        m = re.search(r"network-snapshot-(\d+)\.pkl$", p.name)
        if m and p.stat().st_size > 0:
            out.append((int(m.group(1)), p))
    return sorted(out)

previos = snapshots()
if previos:
    RESUME_KIMG, RESUME_PKL = previos[-1]
    print(f"Se reanuda desde {RESUME_KIMG} kimg → {RESUME_PKL}")
else:
    RESUME_KIMG, RESUME_PKL = 0, None
    print("No hay snapshots previos: entrenamiento nuevo.")
if RESUME_KIMG >= TOTAL_KIMG:
    print(f"✔ Ya se alcanzó el tope ({RESUME_KIMG} ≥ {TOTAL_KIMG} kimg). No hay nada que entrenar.")

No hay snapshots previos: entrenamiento nuevo.


## Paso 6 · Entrenar
La celda muestra una línea de progreso cada ~5 min (kimg, s/kimg, `augment` = probabilidad de ADA). El log completo queda en Drive.
Puedes detenerla en cualquier momento (botón ■): el último snapshot guardado queda en Drive y se reanuda desde él.

In [7]:
env = os.environ.copy()
cmd = [sys.executable, "train.py", f"--outdir={RUN_DIR}", f"--data={SG_ZIP}", "--gpus=1",
       "--cond=1", "--mirror=0", "--metrics=none", "--cfg=auto",
       f"--kimg={TOTAL_KIMG}", f"--snap={SNAP_TICKS}", f"--seed={SEED + RESUME_KIMG}"]
if RESUME_PKL is not None:
    cmd.append(f"--resume={RESUME_PKL}")
    env["SG_RESUME_KIMG"] = str(RESUME_KIMG)

LOG_PATH = LOGS / f"04_gan_{datetime.now().strftime('%Y%m%d_%H%M')}_desde{RESUME_KIMG}kimg.log"
print(" ".join(cmd)); print("Log:", LOG_PATH)

if RESUME_KIMG < TOTAL_KIMG:
    t0 = time.time(); ultima = ""
    with open(LOG_PATH, "w") as flog:
        proc = subprocess.Popen(cmd, cwd=SG_DIR, stdout=flog, stderr=subprocess.STDOUT, text=True, env=env)
        try:
            while proc.poll() is None:
                time.sleep(300 if IN_COLAB else 5)
                ticks = re.findall(r"(tick\s+\d+\s+kimg\s+[\d.]+.*)", LOG_PATH.read_text(errors="ignore"))
                if ticks and ticks[-1] != ultima:
                    ultima = ticks[-1]
                    campos = re.search(r"kimg\s+([\d.]+).*?sec/kimg\s+([\d.]+).*?augment\s+([\d.]+)", ultima)
                    if campos:
                        k, spk, aug = campos.groups()
                        restante = (TOTAL_KIMG - float(k)) * float(spk) / 3600
                        print(f"[{datetime.now().strftime('%H:%M')}] kimg {k} | {spk} s/kimg | ADA p {aug} | faltan ≈ {restante:.1f} h", flush=True)
        except KeyboardInterrupt:
            proc.terminate(); proc.wait(timeout=60)
            print("Detenido manualmente. El último snapshot guardado está en Drive.")
    print(f"Proceso terminado (código {proc.returncode}) tras {(time.time()-t0)/3600:.2f} h")
    print("\n".join(LOG_PATH.read_text(errors="ignore").splitlines()[-8:]))

/usr/bin/python3 train.py --outdir=/content/drive/MyDrive/tesis_retail/checkpoints/stylegan2ada_cond128_seed42 --data=/content/data/train128_stylegan.zip --gpus=1 --cond=1 --mirror=0 --metrics=none --cfg=auto --kimg=2000 --snap=25 --seed=42
Log: /content/drive/MyDrive/tesis_retail/logs/04_gan_20261009_1802_desde0kimg.log
[18:07] kimg 0.0 | 2069.82 s/kimg | ADA p 0.000 | faltan ≈ 1149.9 h
[18:12] kimg 12.0 | 26.21 s/kimg | ADA p 0.018 | faltan ≈ 14.5 h
[18:17] kimg 24.0 | 26.18 s/kimg | ADA p 0.035 | faltan ≈ 14.4 h
[18:22] kimg 36.0 | 26.25 s/kimg | ADA p 0.048 | faltan ≈ 14.3 h
[18:27] kimg 48.0 | 26.36 s/kimg | ADA p 0.057 | faltan ≈ 14.3 h
[18:32] kimg 60.0 | 26.29 s/kimg | ADA p 0.067 | faltan ≈ 14.2 h
[18:37] kimg 72.0 | 26.27 s/kimg | ADA p 0.082 | faltan ≈ 14.1 h
[18:42] kimg 80.0 | 26.36 s/kimg | ADA p 0.091 | faltan ≈ 14.1 h
[18:47] kimg 92.0 | 26.28 s/kimg | ADA p 0.105 | faltan ≈ 13.9 h
[18:52] kimg 104.0 | 26.33 s/kimg | ADA p 0.118 | faltan ≈ 13.9 h
[18:57] kimg 116.0 | 26

## Paso 7 · Resumen y registro

In [ ]:
snaps = snapshots()
print(f"Snapshots en Drive: {len(snaps)} | último: {snaps[-1][0] if snaps else 0} kimg")
for k, p in snaps[-5:]:
    print(f"  {k:>6} kimg  {p.parent.name}/{p.name}  ({p.stat().st_size/2**20:.0f} MB)")
grids = sorted(RUN_DIR.glob("*/fakes*.png"))
if grids:
    print("Última malla de muestras:", grids[-1])
    if IN_COLAB:
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(grids[-1]), width=900))

registro = LOGS / "04_gan_registro.json"
hist = json.loads(registro.read_text()) if registro.exists() else {"sesiones": []}
hist.update({"run": RUN_NAME, "total_kimg_objetivo": TOTAL_KIMG, "snap_kimg": SNAP_KIMG,
             "commits": {"stylegan2_ada": SG_COMMIT, "stylegan3_gradfix": SG3_COMMIT},
             "parches": ["distutils", "Sampler", "grid_sample_gradfix (StyleGAN3)", "custom_ops", "reanudacion kimg/ADA p"],
             "opciones": "--cond=1 --mirror=0 --metrics=none --cfg=auto"})
hist["sesiones"].append({"fin_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                         "desde_kimg": RESUME_KIMG, "hasta_kimg": snaps[-1][0] if snaps else 0,
                         "log": LOG_PATH.name, "versiones": {"python": sys.version.split()[0], "torch": torch.__version__}})
registro.write_text(json.dumps(hist, indent=2, ensure_ascii=False))
print("Registro actualizado:", registro)

Snapshots en Drive: 6 | último: 500 kimg
     100 kimg  00000-train128_stylegan-cond-auto1-kimg2000/network-snapshot-000100.pkl  (278 MB)
     200 kimg  00000-train128_stylegan-cond-auto1-kimg2000/network-snapshot-000200.pkl  (278 MB)
     300 kimg  00000-train128_stylegan-cond-auto1-kimg2000/network-snapshot-000300.pkl  (278 MB)
     400 kimg  00000-train128_stylegan-cond-auto1-kimg2000/network-snapshot-000400.pkl  (278 MB)
     500 kimg  00000-train128_stylegan-cond-auto1-kimg2000/network-snapshot-000500.pkl  (278 MB)
Última malla de muestras: /content/drive/MyDrive/tesis_retail/checkpoints/stylegan2ada_cond128_seed42/00000-train128_stylegan-cond-auto1-kimg2000/fakes_init.png


**Al terminar:** desconecta la GPU (*Entorno de ejecución → Desconectar y borrar*).